# 12. Raster und Karten

Viele Wettbewerbsaufgaben spielen auf einer **Karte** aus Kästchen: ein Feld mit Maulwurfshügeln, ein Land aus schwarzen und
weißen Feldern, eine Landschaft aus Kacheln. In Python ist eine solche Karte ein **Raster**: eine Liste von Zeilen, und
`karte[z][s]` ist das Kästchen in Zeile `z` und Spalte `s`.

In diesem Notebook lernst du:

- wie du eine Rasterdatei einliest, ohne dass Leerzeichen verloren gehen,
- wie du die **Nachbarn** eines Kästchens findest, ohne über den Rand hinauszulaufen,
- wie du ein Raster veränderst und ausgibst,
- wie du ein **Muster** im Raster suchst und dabei nichts doppelt zählst.

Die Beispiele stammen aus den Aufgaben „Kassiopeia“ (2015) und „Baulwürfe“ (2020).

## 1. Ein Raster einlesen

Die Datei `daten/kassiopeia.txt` ist das Beispiel vom Aufgabenblatt „Kassiopeia“: In der ersten Zeile stehen die Anzahl der
Zeilen und der Spalten, darunter das Land. `#` ist ein schwarzes Feld, ein Leerzeichen ein weißes, `K` die Schildkröte
Kassiopeia.

```
6 9
#########
#  #    #
#  # #  #
#  K #  #
#    #  #
#########
```

Beim Einlesen musst du auf die **Leerzeichen** aufpassen, denn sie sind hier Teil der Karte:

- Benutze `splitlines()`, aber **nicht** `split()` oder `strip()`. Die würden Leerzeichen entfernen.
- Manche Editoren löschen Leerzeichen am Zeilenende. Dann ist eine Zeile kürzer als die Karte breit ist, und `karte[z][s]`
  meldet für die letzten Spalten einen `IndexError`. Darum füllst du jede Zeile vorsichtshalber mit Leerzeichen auf die
  richtige Breite auf.

In [ ]:
def lies_raster(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    teile = zeilen[0].split()
    hoehe = int(teile[0])
    breite = int(teile[1])
    karte = []
    for z in range(1, hoehe + 1):
        zeile = zeilen[z]
        zeile = zeile + ' ' * (breite - len(zeile))     # zu kurze Zeilen auffüllen
        karte.append(zeile)
    return karte


karte = lies_raster('daten/kassiopeia.txt')
hoehe = len(karte)
breite = len(karte[0])
print(hoehe, 'Zeilen,', breite, 'Spalten')
for zeile in karte:
    print('|' + zeile + '|')

#### Aufgabe 1.1

Zähle die weißen Felder (Leerzeichen und das Feld, auf dem Kassiopeia steht) und finde Kassiopeias Position: Gib Zeile
und Spalte aus.

**Erwartete Ausgabe:**
```
23 weiße Felder, Kassiopeia steht in Zeile 3 Spalte 3
```

In [ ]:
weiss = 0
kz = -1
ks = -1
# dein Code

<details>
<summary>Tipp anzeigen</summary>

Zwei Schleifen über alle Zeilen `z` und Spalten `s` (Notebook 11). `karte[z][s] in ' K'` wäre eine kurze Schreibweise für
„Leerzeichen oder K“.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
weiss = 0
kz = -1
ks = -1
for z in range(hoehe):
    for s in range(breite):
        if karte[z][s] == ' ' or karte[z][s] == 'K':
            weiss = weiss + 1
        if karte[z][s] == 'K':
            kz = z
            ks = s
print(weiss, 'weiße Felder, Kassiopeia steht in Zeile', kz, 'Spalte', ks)
```

</details>

## 2. Nachbarn

Fast immer geht es um die **Nachbarn** eines Kästchens: Kassiopeia darf nach Norden, Süden, Osten und Westen gehen; beim
Minensuchen zählen auch die vier schrägen Nachbarn. Dabei gibt es eine Gefahr: Am **Rand** gibt es manche Nachbarn nicht.
`karte[z][s + 1]` am rechten Rand meldet einen Fehler, und `karte[z - 1][s]` in Zeile 0 ist noch gemeiner: Wegen der negativen
Position holt Python ohne Fehlermeldung die **letzte** Zeile.

Darum prüfst du vor jedem Zugriff mit einer kleinen Hilfsfunktion, ob das Kästchen im Raster liegt. Die vier Richtungen
schreibst du in zwei Listen. So brauchst du keine vier fast gleichen `if`-Blöcke:

| Richtung | Zeile ändert sich um `dz` | Spalte ändert sich um `ds` |
|---|---|---|
| Norden | -1 | 0 |
| Süden | +1 | 0 |
| Westen | 0 | -1 |
| Osten | 0 | +1 |

In [ ]:
dz = [-1, 1, 0, 0]
ds = [0, 0, -1, 1]


def im_raster(z, s):
    return 0 <= z < hoehe and 0 <= s < breite


def weisse_nachbarn(z, s):
    anzahl = 0
    for r in range(4):
        z2 = z + dz[r]
        s2 = s + ds[r]
        if im_raster(z2, s2) and karte[z2][s2] != '#':
            anzahl = anzahl + 1
    return anzahl


print('Weiße Nachbarn von Kassiopeia:', weisse_nachbarn(3, 3))
print('Weiße Nachbarn der Ecke oben links:', weisse_nachbarn(0, 0))

#### Aufgabe 2.1

Wie viele weiße Felder haben 0, 1, 2, 3 und 4 weiße Nachbarn? Zähle es mit einer Zählliste (Notebook 9) und gib sie aus.
Ein weißes Feld mit genau einem weißen Nachbarn ist eine **Sackgasse**. Wie viele gibt es?

**Erwartete Ausgabe:**
```
0 weiße Felder mit 0 weißen Nachbarn
0 weiße Felder mit 1 weißen Nachbarn
10 weiße Felder mit 2 weißen Nachbarn
12 weiße Felder mit 3 weißen Nachbarn
1 weiße Felder mit 4 weißen Nachbarn
Sackgassen: 0
```

In [ ]:
verteilung = [0] * 5
# dein Code

<details>
<summary>Tipp anzeigen</summary>

Alle Felder durchgehen. Für jedes weiße Feld ist `n = weisse_nachbarn(z, s)` die Position in der Zählliste.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
verteilung = [0] * 5
for z in range(hoehe):
    for s in range(breite):
        if karte[z][s] != '#':
            n = weisse_nachbarn(z, s)
            verteilung[n] = verteilung[n] + 1
for n in range(5):
    print(verteilung[n], 'weiße Felder mit', n, 'weißen Nachbarn')
print('Sackgassen:', verteilung[1])
```

Auf dieser Karte gibt es keine Sackgasse, aber einige enge Gänge mit nur 2 Nachbarn. In der echten Aufgabe geht es darum, ob
Kassiopeia **alle** weißen Felder erreichen kann. Dafür muss man von Feld zu Feld weitergehen. Das lernst du in Notebook 21
(„Erreichbarkeit“).

</details>

Beim **Minensuchen** hat ein Feld acht Nachbarn: dazu kommen die vier schrägen. Statt acht Richtungen aufzuschreiben, kannst
du zwei Schleifen für `dz` und `ds` von -1 bis 1 nehmen und nur den Fall `dz == 0 and ds == 0` (das Feld selbst) auslassen.

Die Datei `daten/minen.txt` (erste Zeile: Zeilen und Spalten, `*` ist eine Mine):

```
5 7
*......
..*....
.....*.
.*.....
....**.
```

#### Aufgabe 2.2

Gib das Minenfeld so aus, wie es das Spiel zeigt: Auf jeder Mine ein `*`, auf jedem anderen Feld die Anzahl der Minen in den
acht Nachbarfeldern, wobei eine 0 als `.` erscheint. Benutze `lies_raster`.

**Erwartete Ausgabe:**
```
*211...
12*1111
12211*1
1*11332
1111**1
```

In [ ]:
minen = lies_raster('daten/minen.txt')
mh = len(minen)
mb = len(minen[0])
# dein Code

<details>
<summary>Tipp anzeigen</summary>

Vier Schleifen ineinander: Zeilen, Spalten, dann `dz8` und `ds8` von -1 bis 1. Zähle eine Mine, wenn das Nachbarfeld nicht das
Feld selbst ist, im Raster liegt und eine Mine enthält. Weil das Minenfeld andere Maße hat als Kassiopeias Karte, prüfst du
den Rand hier direkt mit `mh` und `mb`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
minen = lies_raster('daten/minen.txt')
mh = len(minen)
mb = len(minen[0])
for z in range(mh):
    for s in range(mb):
        if minen[z][s] == '*':
            print('*', end='')
        else:
            anzahl = 0
            for dz8 in range(-1, 2):
                for ds8 in range(-1, 2):
                    z2 = z + dz8
                    s2 = s + ds8
                    if (dz8 != 0 or ds8 != 0) and 0 <= z2 < mh and 0 <= s2 < mb and minen[z2][s2] == '*':
                        anzahl = anzahl + 1
            if anzahl == 0:
                print('.', end='')
            else:
                print(anzahl, end='')
    print()
```

</details>

## 3. Ein Raster verändern

Texte lassen sich nicht verändern (Notebook 4). Willst du Kästchen markieren, wandelst du jede Zeile mit `list(zeile)` in eine
Liste von Zeichen um. Zum Ausgeben setzt du sie mit `''.join(zeile)` wieder zu einem Text zusammen (Notebook 6).

In [ ]:
neu = []
for zeile in karte:
    neu.append(list(zeile))
neu[1][1] = 'o'
for zeile in neu:
    print(''.join(zeile))

#### Aufgabe 3.1

Markiere auf einer Kopie von Kassiopeias Karte jedes weiße Feld mit der Anzahl seiner weißen Nachbarn (Kassiopeias eigenes
Feld bleibt `K`) und gib die Karte aus. So siehst du auf einen Blick Sackgassen (1), Gänge (2) und offene Flächen (3, 4).

**Erwartete Ausgabe:**
```
#########
#22#2232#
#33#2#33#
#34K3#33#
#2332#22#
#########
```

In [ ]:
neu = []
for zeile in karte:
    neu.append(list(zeile))
# dein Code

<details>
<summary>Tipp anzeigen</summary>

Geh alle Felder durch. Bei einem Leerzeichen in der **ursprünglichen** Karte trägst du `str(weisse_nachbarn(z, s))` in `neu`
ein. Lies immer aus `karte` und schreibe in `neu`, dann stören die neuen Ziffern nicht beim Zählen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
neu = []
for zeile in karte:
    neu.append(list(zeile))
for z in range(hoehe):
    for s in range(breite):
        if karte[z][s] == ' ':
            neu[z][s] = str(weisse_nachbarn(z, s))
for zeile in neu:
    print(''.join(zeile))
```

</details>

## 4. Muster im Raster suchen

Ein **Muster** ist selbst ein kleines Raster, zum Beispiel ein 2 × 2-Block aus `X`. Um es zu finden, legst du es gedanklich
auf jede mögliche Stelle der Karte und prüfst, ob alle Kästchen übereinstimmen. Die Stelle ist die Position der linken oberen
Ecke des Musters. Damit das Muster nicht über den Rand ragt, läuft die Zeile nur bis `hoehe - mh` und die Spalte nur bis
`breite - mb` (mit `mh` und `mb` als Höhe und Breite des Musters).

In [ ]:
feld = ['XXX.',
        'XXX.',
        '..XX',
        '..XX']
muster = ['XX',
          'XX']


def passt(feld, muster, z, s):
    for i in range(len(muster)):
        for j in range(len(muster[0])):
            if feld[z + i][s + j] != muster[i][j]:
                return False
    return True


for z in range(len(feld) - len(muster) + 1):
    for s in range(len(feld[0]) - len(muster[0]) + 1):
        if passt(feld, muster, z, s):
            print('Muster gefunden bei Zeile', z, 'Spalte', s)

#### Frage 4.1

Im Feld oben sieht man zwei Blöcke aus `X`. Das Programm findet aber drei Stellen. Warum? Was müsste man tun, wenn man
**Blöcke** zählen will, die sich keine `X` teilen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Der linke Block ist 3 Spalten breit. Das 2 × 2-Muster passt darin an **zwei** Stellen, die sich überlappen (Spalte 0 und
Spalte 1). Will man Blöcke zählen, die sich keine Kästchen teilen, muss man sich die schon benutzten Kästchen **merken** und
eine Fundstelle nur zählen, wenn keines ihrer Kästchen schon benutzt ist. Genau das brauchst du in der Wettbewerbs-Ecke.

</details>

## Wettbewerbs-Ecke: Baulwürfe

*Angelehnt an die Junioraufgabe „Baulwürfe“ (Jugendwettbewerb 2020).*

Auf einer Hügelkarte ist jedes Kästchen mit Maulwurfshügel mit `X` markiert. Eine neue Maulwurfsart, die **Baulwürfe**, baut
ihre Hügel immer in diesem Muster, 3 Spalten breit und 4 Zeilen hoch:

```
XXX
X X
X X
XXX
```

Die Baue grenzen manchmal aneinander, **überlappen sich aber nicht**. Wie viele Baue gibt es auf der Karte?

Die Datei `daten/baulwuerfe.txt` ist das Beispiel `karte5.txt` der echten Aufgabe. Achtung, anderes Format: In der ersten
Zeile steht die **Breite**, in der zweiten die **Höhe**, dann kommt die Karte.

```
14
20
 XXX      XXX 
 X X      X X 
 X XXXXXXXX X 
 XXXX XX XXXX 
XXX X XX X XXX
X X XXXXXX X X
X XXXX  XXXX X
XXXX X  X XXXX
   X X  X X   
   XXX  XXX   
   XXX  XXX   
   X X  X X   
XXXX X  X XXXX
X XXXX  XXXX X
X X XXXXXX X X
XXX X XX X XXX
 XXXX XX XXXX 
 X XXXXXXXX X 
 X X      X X 
 XXX      XXX
```

#### Aufgabe E.1

Lies die Karte ein (mit Auffüllen zu kurzer Zeilen!) und zähle, an wie vielen Stellen das Baulwurf-Muster passt. Benutze die
Funktion `passt` aus Abschnitt 4.

**Erwartete Ausgabe:**
```
20 Stellen, an denen das Muster passt
```

In [ ]:
bau = ['XXX',
       'X X',
       'X X',
       'XXX']
with open('daten/baulwuerfe.txt', encoding='utf-8') as f:
    zeilen = f.read().splitlines()
kb = int(zeilen[0])
kh = int(zeilen[1])
huegel = []
# dein Code

<details>
<summary>Tipp anzeigen</summary>

Die Karte beginnt in `zeilen[2]`. Fülle jede Zeile wie in `lies_raster` auf die Breite `kb` auf. Dann zwei Schleifen über alle
linken oberen Ecken (Zeile bis `kh - 4`, Spalte bis `kb - 3`) und `passt(huegel, bau, z, s)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
bau = ['XXX',
       'X X',
       'X X',
       'XXX']
with open('daten/baulwuerfe.txt', encoding='utf-8') as f:
    zeilen = f.read().splitlines()
kb = int(zeilen[0])
kh = int(zeilen[1])
huegel = []
for z in range(2, kh + 2):
    zeile = zeilen[z]
    huegel.append(zeile + ' ' * (kb - len(zeile)))

funde = 0
for z in range(kh - 4 + 1):
    for s in range(kb - 3 + 1):
        if passt(huegel, bau, z, s):
            funde = funde + 1
print(funde, 'Stellen, an denen das Muster passt')
```

</details>

#### Aufgabe E.2

Die Karte enthält in Wirklichkeit nur 16 Baue. Stoßen Baue aneinander, kann das Muster auch an Stellen passen, die aus Teilen
**mehrerer** Baue bestehen. Ein solcher „Bau“ würde Hügel mit anderen Bauen teilen. Zähle deshalb eine Fundstelle nur dann, wenn
keiner ihrer Hügel schon zu einem gezählten Bau gehört, und markiere dann ihre Hügel in einer Tabelle `benutzt`.

**Erwartete Ausgabe:**
```
16 Baue
```

In [ ]:
benutzt = []
for z in range(kh):
    benutzt.append([False] * kb)
baue = 0
# dein Code
print(baue, 'Baue')

<details>
<summary>Tipp anzeigen</summary>

Eine Hilfsfunktion `schon_benutzt(z, s)` prüft für alle `X`-Kästchen des Musters, ob `benutzt[z + i][s + j]` schon wahr ist.
Wird ein Bau gezählt, setzt du für alle seine `X`-Kästchen `benutzt[z + i][s + j] = True`. Die Tabelle legst du ohne die Falle
aus Notebook 11 an (steht schon im Gerüst).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
benutzt = []
for z in range(kh):
    benutzt.append([False] * kb)
baue = 0


def schon_benutzt(z, s):
    for i in range(4):
        for j in range(3):
            if bau[i][j] == 'X' and benutzt[z + i][s + j]:
                return True
    return False


for z in range(kh - 4 + 1):
    for s in range(kb - 3 + 1):
        if passt(huegel, bau, z, s) and not schon_benutzt(z, s):
            baue = baue + 1
            for i in range(4):
                for j in range(3):
                    if bau[i][j] == 'X':
                        benutzt[z + i][s + j] = True
print(baue, 'Baue')
```

Mit dem Merken kommen genau die 16 Baue heraus, die auch die offiziellen Lösungshinweise nennen. Die 4 zusätzlichen Fundstellen
aus E.1 sind „scheinbare“ Baue aus Teilen ihrer Nachbarn. Ob man sie mitzählt, hängt davon ab, wie man „überlappen sich nicht“
liest: wieder eine Entscheidung für die Dokumentation (Notebook 23). Die vollständige Lösung mit allen Karten und Bildern der
Fundstellen steht in `jwinf3/39_Baulwuerfe/baulwuerfe_loesung.ipynb`.

</details>

## Das hast du gelernt

| Aufgabe | So geht es |
|---|---|
| Raster einlesen | `splitlines()`, **nicht** `split()` oder `strip()`; zu kurze Zeilen mit Leerzeichen auffüllen |
| Kästchen lesen | `karte[z][s]` (Zeile, Spalte) |
| im Raster? | `0 <= z < hoehe and 0 <= s < breite`, als Hilfsfunktion |
| vier Nachbarn | Listen `dz = [-1, 1, 0, 0]`, `ds = [0, 0, -1, 1]` |
| acht Nachbarn | zwei Schleifen von -1 bis 1, ohne (0, 0) |
| Raster verändern | Zeilen mit `list(zeile)` umwandeln, mit `''.join(zeile)` ausgeben |
| Muster suchen | jede linke obere Ecke ausprobieren, Grenzen `hoehe - mh` und `breite - mb` |
| nichts doppelt zählen | benutzte Kästchen in einer Tabelle merken |

- Vorsicht bei negativen Positionen: `karte[-1]` ist die letzte Zeile, ohne Fehlermeldung.
- Aus der alten Karte lesen, in eine neue schreiben.

Im nächsten Notebook lässt du Dinge auf solchen Karten ablaufen: Simulationen Schritt für Schritt.